# Fine-tune BiomedCLIP on ROCOv2 - Kaggle T4 x2

Notebook Kaggle autonome pour fine-tuner BiomedCLIP sur ROCOv2 avec Hugging Face pour le modele et les donnees.

- Accelerator Kaggle recommande: GPU T4 x2.
- Donnees: `eltorio/ROCOv2-radiology` via `datasets`.
- Modele: `microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224` via `open_clip` + Hugging Face Hub.
- Entrainement: contrastive CLIP image-caption, AMP fp16, `DataParallel`, gradient accumulation.
- Learning rate volontairement bas et stable: `1e-5` pour un full fine-tuning.
- Sortie finale: dossier et zip dans `/kaggle/working/biomedclip-rocov2-finetuned`.

Active Internet dans Kaggle avant de lancer le notebook.

## A remplir si besoin

- `HF_REPO_ID`: uniquement si `PUSH_TO_HUB = True`, mets ton repo Hugging Face, par exemple `ton-user/biomedclip-rocov2-finetuned`.
- `HF_TOKEN`: pas obligatoire pour telecharger ce modele/dataset publics. Ajoute-le dans Kaggle Secrets seulement si tu veux push vers Hugging Face ou eviter les limites d'acces.
- `EPOCHS`: 4 par defaut pour un run final prudent. Evite >5 sans validation.
- `BATCH_SIZE`: 16 par defaut, safe pour T4 x2. Essaie 32 seulement si la VRAM tient.
- `LR`: 1e-5 par defaut, volontairement bas pour ne pas casser BiomedCLIP.
- `RUN_SMOKE_BATCH`: laisse `True` au premier lancement. Ca verifie un batch avant le vrai training.

In [ ]:
# Installation Kaggle-safe.
# Important: on n'installe PAS datasets avec ses dependances, sinon pip peut downgrader fsspec
# et casser les packages preinstalles Kaggle comme s3fs/gcsfs.
!pip -q install -U "open_clip_torch==2.32.0" "huggingface_hub==0.36.2" "transformers==4.57.6" "safetensors>=0.4" "hf_xet>=1.1.0"

import importlib.util
import subprocess
import sys

if importlib.util.find_spec("datasets") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "datasets>=4.0,<5.0"])
else:
    print("datasets deja installe: on garde la version Kaggle pour eviter les conflits fsspec")

In [ ]:
import json
import math
import os
import random
import shutil
import time
from io import BytesIO
from pathlib import Path

# Evite de remplir /kaggle/working avec le cache Hugging Face; garde /kaggle/working pour le modele final.
os.environ.setdefault("HF_HOME", "/kaggle/temp/huggingface")
os.environ.setdefault("HF_DATASETS_CACHE", "/kaggle/temp/huggingface/datasets")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import numpy as np
import open_clip
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import HfApi, dataset_info, hf_hub_download, login, model_info, notebook_login
from PIL import Image
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

MODEL_REPO = "microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"
MODEL_ID = f"hf-hub:{MODEL_REPO}"
DATASET_ID = "eltorio/ROCOv2-radiology"

OUTPUT_DIR = Path("/kaggle/working/biomedclip-rocov2-finetuned")
SEED = 42

# Bon point de depart pour full fine-tuning CLIP medical: bas, stable, peu agressif.
EPOCHS = 4
LR = 1e-5
WEIGHT_DECAY = 0.02
WARMUP_RATIO = 0.05
GRAD_CLIP_NORM = 1.0

# T4 x2: DataParallel split ce batch global sur les deux GPUs.
# 16 est volontairement safe sur T4 16GB; augmente a 32 seulement si tu as de la marge VRAM.
CONTEXT_LENGTH = 256
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 2

# Laisse None pour tout ROCOv2 train. Mets une valeur seulement pour debug rapide.
MAX_TRAIN_SAMPLES = None

# Pas de test lourd ici. Optionnel: val loss sur quelques batches si tu veux choisir un checkpoint.
RUN_SMOKE_BATCH = True
RUN_VALIDATION = False
VAL_BATCHES = 100

# Optionnel: push le dossier final vers ton compte Hugging Face.
PUSH_TO_HUB = False
HF_REPO_ID = "Ozantsk/biomedclip-rocov2-finetuned"

In [ ]:
# Optionnel: utilise un token Hugging Face stocke dans Kaggle Secrets sous le nom HF_TOKEN.
# Pas necessaire pour telecharger BiomedCLIP/ROCOv2 s'ils restent publics.
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("HF_TOKEN detecte: login Hugging Face OK")
else:
    print("Aucun HF_TOKEN: OK pour telecharger des repos publics. Token requis seulement pour push ou repos prives/gated.")

In [ ]:
# Smoke-check leger: si cette cellule echoue sur Kaggle, active Internet ou verifie le token.
print("checking model repo...")
print(model_info(MODEL_REPO).modelId)
print("checking dataset repo...")
print(dataset_info(DATASET_ID).id)
print("Hugging Face access OK")

In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()

if NUM_GPUS < 2:
    BATCH_SIZE = min(BATCH_SIZE, 16)
    print("Moins de 2 GPUs detectes: batch size reduit automatiquement a", BATCH_SIZE)

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("device:", DEVICE)
print("num_gpus:", NUM_GPUS)
for idx in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(idx)
    print(f"gpu {idx}: {props.name} - {props.total_memory / 1024**3:.1f} GB")

effective_batch = BATCH_SIZE * GRAD_ACCUM_STEPS
print("batch_size:", BATCH_SIZE)
print("grad_accum_steps:", GRAD_ACCUM_STEPS)
print("effective_batch_size:", effective_batch)
print("learning_rate:", LR)

In [ ]:
# Tout vient de Hugging Face. Le dataset fait environ 18.6 GB.
raw_ds = load_dataset(DATASET_ID)
train_ds = raw_ds["train"]
val_ds = raw_ds["validation"] if "validation" in raw_ds else None

if MAX_TRAIN_SAMPLES is not None:
    n = min(int(MAX_TRAIN_SAMPLES), len(train_ds))
    train_ds = train_ds.shuffle(seed=SEED).select(range(n))

print(raw_ds)
print("columns:", train_ds.column_names)
print(f"train rows: {len(train_ds):,}")
if val_ds is not None:
    print(f"validation rows: {len(val_ds):,}")

sample = train_ds[0]
print("sample image_id:", sample.get("image_id"))
print("sample caption:", sample.get("caption"))

In [ ]:
# Chargement Hugging Face Hub via open_clip.
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(MODEL_ID)
tokenizer = open_clip.get_tokenizer(MODEL_ID)

# Pour la radiologie, on evite les augmentations type flip aleatoire gauche/droite.
preprocess = preprocess_val

model = model.to(DEVICE)
if NUM_GPUS >= 2:
    model = torch.nn.DataParallel(model)
    print("DataParallel active sur", NUM_GPUS, "GPUs")
else:
    print("DataParallel non active")

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"params total: {total_params:,}")
print(f"params trainable: {trainable_params:,}")

In [ ]:
def clean_caption(text) -> str:
    if text is None:
        return ""
    return " ".join(str(text).split())


def load_pil_image(value):
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, dict):
        if value.get("bytes") is not None:
            return Image.open(BytesIO(value["bytes"])).convert("RGB")
        if value.get("path") is not None:
            return Image.open(value["path"]).convert("RGB")
    return Image.open(value).convert("RGB")


def collate_roco(batch):
    images = []
    captions = []
    image_ids = []

    for item in batch:
        caption = clean_caption(item.get("caption"))
        if not caption or item.get("image") is None:
            continue
        try:
            image = load_pil_image(item["image"])
            images.append(preprocess(image))
            captions.append(caption)
            image_ids.append(item.get("image_id", ""))
        except Exception:
            # Quelques fichiers corrompus ne doivent pas stopper tout le fine-tuning.
            continue

    if not images:
        return None

    image_tensor = torch.stack(images)
    text_tensor = tokenizer(captions, context_length=CONTEXT_LENGTH)
    return image_tensor, text_tensor, captions, image_ids


train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
    collate_fn=collate_roco,
    drop_last=True,
)

val_loader = None
if RUN_VALIDATION and val_ds is not None:
    val_loader = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=NUM_WORKERS > 0,
        collate_fn=collate_roco,
        drop_last=False,
    )

print("train batches:", len(train_loader))
if val_loader is not None:
    print("validation batches:", len(val_loader))

In [ ]:
def unwrap_model(m):
    return m.module if isinstance(m, torch.nn.DataParallel) else m


def forward_clip(m, images, texts):
    out = m(images, texts)
    if isinstance(out, dict):
        image_features = out["image_features"]
        text_features = out["text_features"]
        logit_scale = out["logit_scale"]
    else:
        image_features, text_features, logit_scale = out[:3]

    # DataParallel peut renvoyer une valeur par GPU pour un scalaire.
    if isinstance(logit_scale, torch.Tensor) and logit_scale.ndim > 0:
        logit_scale = logit_scale.mean()
    return image_features, text_features, logit_scale


def clip_contrastive_loss(image_features, text_features, logit_scale):
    logits_per_image = logit_scale * (image_features @ text_features.t())
    logits_per_text = logits_per_image.t()
    labels = torch.arange(logits_per_image.shape[0], device=logits_per_image.device)
    image_loss = F.cross_entropy(logits_per_image, labels)
    text_loss = F.cross_entropy(logits_per_text, labels)
    return (image_loss + text_loss) / 2


def build_optimizer(m):
    decay = []
    no_decay = []
    for name, param in unwrap_model(m).named_parameters():
        if not param.requires_grad:
            continue
        lname = name.lower()
        if param.ndim < 2 or "bias" in lname or "norm" in lname or "ln" in lname or "bn" in lname or "logit_scale" in lname:
            no_decay.append(param)
        else:
            decay.append(param)

    return torch.optim.AdamW(
        [
            {"params": decay, "weight_decay": WEIGHT_DECAY},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=LR,
        betas=(0.9, 0.98),
        eps=1e-6,
    )


optimizer = build_optimizer(model)

updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_updates = max(1, EPOCHS * updates_per_epoch)
warmup_steps = max(1, int(total_updates * WARMUP_RATIO))


def lr_schedule(step):
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = float(step - warmup_steps) / float(max(1, total_updates - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)
use_amp = DEVICE.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

print("updates_per_epoch:", updates_per_epoch)
print("total_updates:", total_updates)
print("warmup_steps:", warmup_steps)

In [ ]:
def save_openclip_checkpoint(output_dir: Path, epoch: int, global_step: int) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    model_to_save = unwrap_model(model).cpu()

    torch.save(model_to_save.state_dict(), output_dir / "open_clip_pytorch_model.bin")
    model_to_save.to(DEVICE)

    config_path = hf_hub_download(repo_id=MODEL_REPO, filename="open_clip_config.json")
    shutil.copy(config_path, output_dir / "open_clip_config.json")

    training_config = {
        "base_model": MODEL_REPO,
        "dataset": DATASET_ID,
        "epoch": epoch,
        "global_step": global_step,
        "learning_rate": LR,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "effective_batch_size": BATCH_SIZE * GRAD_ACCUM_STEPS,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "context_length": CONTEXT_LENGTH,
    }
    (output_dir / "training_config.json").write_text(json.dumps(training_config, indent=2))

    readme = f"""---
library_name: open_clip
base_model: {MODEL_REPO}
datasets:
- {DATASET_ID}
tags:
- biomedclip
- rocov2
- medical-image-retrieval
---

# BiomedCLIP fine-tuned on ROCOv2

Fine-tuned from `{MODEL_REPO}` on `{DATASET_ID}` with CLIP contrastive image-caption loss.

Main hyperparameters:
- LR: {LR}
- Epochs: {EPOCHS}
- Batch size: {BATCH_SIZE}
- Gradient accumulation: {GRAD_ACCUM_STEPS}
- Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}
- Context length: {CONTEXT_LENGTH}

Saved files:
- `open_clip_pytorch_model.bin`
- `open_clip_config.json`
- `training_config.json`
"""
    (output_dir / "README.md").write_text(readme)


@torch.no_grad()
def run_validation(max_batches: int = 100):
    if val_loader is None:
        return None
    model.eval()
    losses = []
    for batch_idx, batch in enumerate(tqdm(val_loader, desc="validation")):
        if max_batches is not None and batch_idx >= max_batches:
            break
        if batch is None:
            continue
        images, texts, _, _ = batch
        images = images.to(DEVICE, non_blocking=True)
        texts = texts.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            image_features, text_features, logit_scale = forward_clip(model, images, texts)
            loss = clip_contrastive_loss(image_features, text_features, logit_scale)
        losses.append(float(loss.detach().cpu()))
    model.train()
    return float(np.mean(losses)) if losses else None

In [ ]:
# Verification tres legere avant de lancer plusieurs heures de training.
# Si cette cellule passe, le modele, le tokenizer, le preprocess et le dataset chargent ensemble.
if RUN_SMOKE_BATCH:
    model.eval()
    smoke_batch = None
    smoke_iter = iter(train_loader)
    for _ in range(10):
        candidate = next(smoke_iter)
        if candidate is not None:
            smoke_batch = candidate
            break

    if smoke_batch is None:
        raise RuntimeError("Impossible de construire un batch ROCOv2 valide. Verifie le chargement du dataset.")

    images, texts, captions, image_ids = smoke_batch
    images = images.to(DEVICE, non_blocking=True)
    texts = texts.to(DEVICE, non_blocking=True)

    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=use_amp):
            image_features, text_features, logit_scale = forward_clip(model, images, texts)
            smoke_loss = clip_contrastive_loss(image_features, text_features, logit_scale)

    print("smoke batch OK")
    print("images:", tuple(images.shape))
    print("texts:", tuple(texts.shape))
    print("image_features:", tuple(image_features.shape))
    print("text_features:", tuple(text_features.shape))
    print("loss:", float(smoke_loss.detach().cpu()))
    print("example caption:", captions[0][:220])

    del smoke_batch, smoke_iter, images, texts, image_features, text_features, smoke_loss
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    model.train()
else:
    print("RUN_SMOKE_BATCH=False: smoke-check saute.")

In [ ]:
global_step = 0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    micro_batches = 0
    pending_grad = False

    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}")
    for batch_idx, batch in enumerate(pbar):
        if batch is None:
            continue

        images, texts, _, _ = batch
        images = images.to(DEVICE, non_blocking=True)
        texts = texts.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            image_features, text_features, logit_scale = forward_clip(model, images, texts)
            loss = clip_contrastive_loss(image_features, text_features, logit_scale)
            scaled_loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(scaled_loss).backward()
        running_loss += float(loss.detach().cpu())
        micro_batches += 1
        pending_grad = True

        should_step = micro_batches % GRAD_ACCUM_STEPS == 0
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(unwrap_model(model).parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            with torch.no_grad():
                unwrap_model(model).logit_scale.clamp_(0, math.log(100))
            scheduler.step()
            global_step += 1
            pending_grad = False

        avg_loss = running_loss / max(1, micro_batches)
        pbar.set_postfix(loss=f"{avg_loss:.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}", step=global_step)

    if pending_grad:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(unwrap_model(model).parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        with torch.no_grad():
            unwrap_model(model).logit_scale.clamp_(0, math.log(100))
        scheduler.step()
        global_step += 1

    train_loss = running_loss / max(1, micro_batches)
    val_loss = run_validation(VAL_BATCHES) if RUN_VALIDATION else None
    if val_loss is None:
        print(f"epoch {epoch} done - train_loss={train_loss:.4f} - global_step={global_step}")
    else:
        print(f"epoch {epoch} done - train_loss={train_loss:.4f} - val_loss={val_loss:.4f} - global_step={global_step}")

    save_openclip_checkpoint(OUTPUT_DIR, epoch=epoch, global_step=global_step)
    print("checkpoint saved to", OUTPUT_DIR)

elapsed = (time.time() - start_time) / 60
print(f"training finished in {elapsed:.1f} minutes")

In [ ]:
# Sauvegarde finale + zip telechargeable dans les outputs Kaggle.
save_openclip_checkpoint(OUTPUT_DIR, epoch=EPOCHS, global_step=global_step)
zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)

print("final model folder:", OUTPUT_DIR)
print("final model zip:", zip_path)
print("files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print("-", path.name)

if PUSH_TO_HUB:
    if not os.environ.get("HF_TOKEN"):
        notebook_login()
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    api.upload_folder(repo_id=HF_REPO_ID, repo_type="model", folder_path=str(OUTPUT_DIR))
    print("uploaded to:", HF_REPO_ID)

## Recharger le modele fine-tune localement

Exemple minimal si tu veux recharger le checkpoint sauvegarde dans un autre notebook/script:

```python
import json
from pathlib import Path
import open_clip
from open_clip.factory import _MODEL_CONFIGS

ckpt_dir = Path('/kaggle/working/biomedclip-rocov2-finetuned')
with open(ckpt_dir / 'open_clip_config.json') as f:
    config = json.load(f)

model_name = 'biomedclip_rocov2_finetuned'
_MODEL_CONFIGS[model_name] = config['model_cfg']

model, _, preprocess = open_clip.create_model_and_transforms(
    model_name=model_name,
    pretrained=str(ckpt_dir / 'open_clip_pytorch_model.bin'),
    **{f'image_{k}': v for k, v in config['preprocess_cfg'].items()},
)
tokenizer = open_clip.get_tokenizer(model_name)
```
